# 01_baseline_hybrid_rule

Notebook tạo baseline dự báo `Quantity` cho 56 ngày tiếp theo theo từng SKU.

**Ý tưởng chính:** không dùng ML, chỉ dùng rule thống kê từ quá khứ:
- SKU không bán gần đây → dự báo 0.
- SKU bán rất thưa → dự báo trung bình thấp theo 90 ngày.
- SKU bán tương đối đều → dùng moving average.
- SKU top profit → ưu tiên thêm weekday average.

**Output:** `submission_baseline_hybrid_v01.csv`


# Cell 1 — Import thư viện

Import các thư viện cần dùng:
- `pandas`: xử lý bảng dữ liệu.
- `numpy`: tính toán số học.
- `Path`: quản lý đường dẫn file.

Baseline này chưa dùng model ML.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Cell 2 — Khai báo cấu hình chung

Cell này gom các tham số chính của baseline ở một chỗ.

Có thể chỉnh nhanh:
- `FORECAST_HORIZON`: số ngày cần dự báo.
- `WINDOW_28`, `WINDOW_56`, `WINDOW_90`: các khoảng thời gian lấy trung bình.
- `SPARSE_SALES_DAYS_THRESHOLD`: ngưỡng SKU bán thưa.
- `TOP_PROFIT_QUANTILE`: ngưỡng chọn SKU top profit.
- `SUBMISSION_FILE_NAME`: tên file nộp.


In [2]:
# Số ngày cần dự báo: 28 ngày Public + 28 ngày Private = 56 ngày
FORECAST_HORIZON = 56
PUBLIC_HORIZON = 28

# Các cửa sổ thống kê quá khứ
WINDOW_28 = 28
WINDOW_56 = 56
WINDOW_90 = 90
WEEKDAY_WINDOW = 56  # 8 tuần = 56 ngày

# Rule phân nhóm SKU
SPARSE_SALES_DAYS_THRESHOLD = 3   # SKU bán 1-3 ngày trong 90 ngày được xem là bán rất thưa
TOP_PROFIT_QUANTILE = 0.80        # Top 20% SKU theo profit được xem là nhóm quan trọng

# Tên file output
SUBMISSION_FILE_NAME = "submission_baseline_hybrid_v01.csv"

# Cell 3 — import file dữ liệu

Cell này nhập `train.csv` và `sample_submission.csv` 

In [27]:
DATA_DIR = Path("dataset")

TRAIN_PATH = DATA_DIR / "train_cleaned_01.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBMISSION_PATH = OUTPUT_DIR / SUBMISSION_FILE_NAME

print("TRAIN_PATH:", TRAIN_PATH)
print("SAMPLE_SUB_PATH:", SAMPLE_SUB_PATH)
print("SUBMISSION_PATH:", SUBMISSION_PATH)

TRAIN_PATH: dataset\train_cleaned_01.csv
SAMPLE_SUB_PATH: dataset\sample_submission.csv
SUBMISSION_PATH: outputs\submission_baseline_hybrid_v01.csv


# Cell 4 — Đọc dữ liệu

Đọc 2 file chính:
- `train.csv`: lịch sử giao dịch.
- `sample_submission.csv`: mẫu định dạng nộp bài.

Sau khi chạy, kiểm tra `shape` và vài dòng đầu để chắc chắn file đã đọc đúng.


In [4]:
train = pd.read_csv(TRAIN_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("Train shape:", train.shape)
print("Sample submission shape:", sample_sub.shape)

print("\nTrain preview:")
display(train.head())

print("\nSample submission preview:")
display(sample_sub.head())

Train shape: (710188, 8)
Sample submission shape: (31944, 29)

Train preview:


C:\Users\Louis\AppData\Local\Temp\ipykernel_29188\2711858823.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(TRAIN_PATH)


,Date,Stt,ItemCode,Quantity,UnitPrice,SalesAmount,Unit_Cost,Cost_Amount
0,2020-11-17,2000004,SKU-08063,12,242700.0000,2184300,123559.1,1482709.0
1,2020-11-17,2000003,SKU-09458,600,131818.1818,79090909,110000.0,66000000.0
2,2020-11-18,2000007,SKU-08062,6,230000.0000,940909,101000.0,606000.0
3,2020-11-18,2000006,SKU-09458,240,270000.0000,44181818,110000.0,26400000.0
4,2020-11-18,2000005,SKU-09458,240,270000.0000,44181818,110000.0,26400000.0



Sample submission preview:


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,SKU-00002_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,SKU-00003_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,SKU-00004_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,SKU-00005_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Cell 5 — Kiểm tra nhanh cấu trúc dữ liệu

Kiểm tra trước khi xử lý sâu:
- Tên cột.
- Missing values.
- Số SKU trong train.
- Số dòng trong sample submission.

Mục tiêu là phát hiện sớm lỗi file hoặc sai format.


In [5]:
print("Columns in train:")
print(train.columns.tolist())

print("\nColumns in sample submission:")
print(sample_sub.columns.tolist())

print("\nMissing values in train:")
display(train.isna().sum())

print("\nMissing values in sample submission:")
display(sample_sub.isna().sum())

print("\nNumber of unique SKUs in train:", train["ItemCode"].nunique())
print("Number of rows in sample submission:", len(sample_sub))

Columns in train:
['Date', 'Stt', 'ItemCode', 'Quantity', 'UnitPrice', 'SalesAmount', 'Unit_Cost', 'Cost_Amount']

Columns in sample submission:
['id', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19', 'F20', 'F21', 'F22', 'F23', 'F24', 'F25', 'F26', 'F27', 'F28']

Missing values in train:


Date              0
Stt               0
ItemCode          0
Quantity          0
UnitPrice         0
SalesAmount       0
Unit_Cost         0
Cost_Amount    8881
dtype: int64


Missing values in sample submission:


id     0
F1     0
F2     0
F3     0
F4     0
F5     0
F6     0
F7     0
F8     0
F9     0
F10    0
F11    0
F12    0
F13    0
F14    0
F15    0
F16    0
F17    0
F18    0
F19    0
F20    0
F21    0
F22    0
F23    0
F24    0
F25    0
F26    0
F27    0
F28    0
dtype: int64


Number of unique SKUs in train: 15971
Number of rows in sample submission: 31944


# Cell 6 — Chuẩn hóa ngày và tạo Profit

Cell này xử lý các cột nền tảng:
- Chuyển `Date` sang kiểu ngày tháng.
- Chuyển `Quantity`, `SalesAmount`, `Cost Amount` sang dạng số.
- Tạo `Profit = SalesAmount - Cost Amount`.

`Profit` dùng để nhận diện SKU quan trọng, vì metric của đề có trọng số theo lợi nhuận.


In [6]:
train["Date"] = pd.to_datetime(train["Date"])

# Hàm chuyển cột về dạng số.
# Một số cột trong dữ liệu có thể bị đọc thành text do có dấu phẩy, dấu cách hoặc ký tự lạ.
def to_number(series):
    return (
        series
        .astype(str)
        .str.replace(",", ".", regex=False)      # đổi dấu phẩy thập phân thành dấu chấm nếu có
        .str.replace(" ", "", regex=False)       # bỏ khoảng trắng
        .str.replace("\u00a0", "", regex=False)  # bỏ non-breaking space nếu có
        .pipe(pd.to_numeric, errors="coerce")    # chuyển sang số, lỗi thì thành NaN
    )

numeric_cols = ["Quantity", "SalesAmount", "Cost_Amount"]

for col in numeric_cols:
    train[col] = to_number(train[col])

# Nếu sau khi convert có NaN, thay bằng 0 để tránh lỗi tính toán
train[numeric_cols] = train[numeric_cols].fillna(0)

# Tạo Profit = doanh thu - chi phí
train["Profit"] = train["SalesAmount"] - train["Cost_Amount"]

print("Data types after conversion:")
display(train[numeric_cols + ["Profit"]].dtypes)

print("\nMissing values after conversion:")
display(train[numeric_cols + ["Profit"]].isna().sum())

print("\nDate range:")
print(train["Date"].min(), "→", train["Date"].max())

display(train.head())

Data types after conversion:


Quantity         int64
SalesAmount      int64
Cost_Amount    float64
Profit         float64
dtype: object


Missing values after conversion:


Quantity       0
SalesAmount    0
Cost_Amount    0
Profit         0
dtype: int64


Date range:
2020-11-17 00:00:00 → 2025-09-05 00:00:00


,Date,Stt,ItemCode,Quantity,UnitPrice,SalesAmount,Unit_Cost,Cost_Amount,Profit
0,2020-11-17,2000004,SKU-08063,12,242700.0000,2184300,123559.1,1482709.0,701591.0
1,2020-11-17,2000003,SKU-09458,600,131818.1818,79090909,110000.0,66000000.0,13090909.0
2,2020-11-18,2000007,SKU-08062,6,230000.0000,940909,101000.0,606000.0,334909.0
3,2020-11-18,2000006,SKU-09458,240,270000.0000,44181818,110000.0,26400000.0,17781818.0
4,2020-11-18,2000005,SKU-09458,240,270000.0000,44181818,110000.0,26400000.0,17781818.0


# Cell 7 — Kiểm tra return transaction

Kiểm tra các dòng trả hàng, thường có:
- `Quantity < 0`
- `SalesAmount < 0`
- `Cost Amount < 0`

Baseline chỉ kiểm tra để hiểu dữ liệu. Việc xử lý chính sẽ nằm ở bước gom daily sales và clip `Quantity` âm về 0.


In [7]:
return_mask = (
    (train["Quantity"] < 0) &
    (train["SalesAmount"] < 0) &
    (train["Cost_Amount"] < 0)
)

print("Number of return rows:", int(return_mask.sum()))
print("Return ratio:", round(float(return_mask.mean()), 6))

print("\nReturn transaction preview:")
display(train.loc[return_mask].head())

Number of return rows: 34708
Return ratio: 0.048872

Return transaction preview:


,Date,Stt,ItemCode,Quantity,UnitPrice,SalesAmount,Unit_Cost,Cost_Amount,Profit
47,2020-12-09,2000110,SKU-08063,-12,-209102.2500,-2509227,0.00,-1482709.0,-1026518.0
48,2020-12-09,2000110,SKU-08061,-12,-231818.1667,-2781818,0.00,-1897349.0,-884469.0
1173,2021-05-21,2001755,SKU-07770,-18,-231818.1667,-4172727,-174000.00,-3132000.0,-1040727.0
1305,2021-06-08,2002157,SKU-02768,-24,-161000.0000,-3864000,-85000.00,-2040000.0,-1824000.0
1309,2021-06-08,2002162,SKU-10532,-2,-676618.0000,-1353236,-390065.25,-780131.0,-573105.0


# Cell 8 — Lấy danh sách SKU từ sample_submission

Lấy danh sách SKU cần dự báo từ `sample_submission.csv`.

Lý do: Kaggle yêu cầu nộp đúng danh sách `id` trong sample submission. Lấy từ file mẫu giúp tránh thiếu hoặc thừa SKU.


In [8]:
# id có dạng: SKU-00001_validation hoặc SKU-00001_evaluation
sample_ids = sample_sub["id"].astype(str)

# Xóa hậu tố _validation hoặc _evaluation để lấy ItemCode gốc
all_skus = (
    sample_ids
    .str.replace(r"_(validation|evaluation)$", "", regex=True)
    .unique()
)

all_skus = pd.Index(all_skus, name="ItemCode")

print("Number of SKUs from sample submission:", len(all_skus))
print("First 5 SKUs:", list(all_skus[:5]))

Number of SKUs from sample submission: 15972
First 5 SKUs: ['SKU-00001', 'SKU-00002', 'SKU-00003', 'SKU-00004', 'SKU-00005']


# Cell 9 — Gom transaction thành daily sales theo SKU

Dữ liệu gốc là transaction-level, một SKU trong một ngày có thể có nhiều dòng.

Cell này gom về daily-level:

```text
Date x ItemCode → tổng Quantity, SalesAmount, Cost Amount, Profit
```

Sau khi gom, `Quantity` âm được đưa về 0 để baseline dự báo nhu cầu bán không âm.


In [9]:
daily = (
    train
    .groupby(["Date", "ItemCode"], as_index=False)
    .agg({
        "Quantity": "sum",
        "SalesAmount": "sum",
        "Cost_Amount": "sum",
        "Profit": "sum",
    })
)

# Baseline dự báo nhu cầu bán ra, nên không để Quantity âm
daily["Quantity"] = daily["Quantity"].clip(lower=0)

print("Daily shape:", daily.shape)
display(daily.head())

Daily shape: (506884, 6)


,Date,ItemCode,Quantity,SalesAmount,Cost_Amount,Profit
0,2020-11-17,SKU-08063,12,2184300,1482709.0,701591.0
1,2020-11-17,SKU-09458,600,79090909,66000000.0,13090909.0
2,2020-11-18,SKU-08062,6,940909,606000.0,334909.0
3,2020-11-18,SKU-09458,480,88363636,52800000.0,35563636.0
4,2020-11-20,SKU-09458,240,44181818,26400000.0,17781818.0


# Cell 10 — Hàm tạo dữ liệu đủ ngày trong window gần nhất

Tạo hàm hỗ trợ để sinh bảng đủ:

```text
Date x ItemCode
```

trong một khoảng gần nhất, ví dụ 28, 56 hoặc 90 ngày.

Cách này nhẹ hơn tạo full table cho toàn bộ 5 năm dữ liệu.


In [10]:
def make_complete_recent_window(daily_data, all_skus, end_date, window_days):
    """
    Tạo bảng đủ Date x ItemCode trong window gần nhất.
    
    Parameters
    ----------
    daily_data : DataFrame
        Bảng daily đã group theo Date x ItemCode.
    all_skus : Index hoặc list
        Danh sách SKU cần dự báo.
    end_date : Timestamp
        Ngày cuối cùng của train.
    window_days : int
        Số ngày muốn lấy, ví dụ 28, 56, 90.
    
    Returns
    -------
    DataFrame gồm Date, ItemCode, Quantity.
    Nếu một SKU không bán trong ngày nào đó, Quantity = 0.
    """
    start_date = end_date - pd.Timedelta(days=window_days - 1)
    dates = pd.date_range(start=start_date, end=end_date, freq="D")
    
    full_index = pd.MultiIndex.from_product(
        [dates, all_skus],
        names=["Date", "ItemCode"]
    )
    
    window = (
        daily_data.loc[
            (daily_data["Date"] >= start_date) &
            (daily_data["Date"] <= end_date),
            ["Date", "ItemCode", "Quantity"]
        ]
        .set_index(["Date", "ItemCode"])
        .reindex(full_index)
        .reset_index()
    )
    
    window["Quantity"] = window["Quantity"].fillna(0)
    return window

# Cell 11 — Xác định ngày cuối train và 56 ngày tương lai

Cell này xác định:
- Ngày cuối cùng trong train.
- 56 ngày cần dự báo sau ngày cuối train.

56 ngày này sẽ được chia thành:
- 28 ngày đầu: `_validation`.
- 28 ngày sau: `_evaluation`.


In [11]:
last_train_date = daily["Date"].max()

future_dates = pd.date_range(
    start=last_train_date + pd.Timedelta(days=1),
    periods=FORECAST_HORIZON,
    freq="D"
)

print("Last train date:", last_train_date.date())
print("Future start:", future_dates[0].date())
print("Future end:", future_dates[-1].date())
print("Number of future days:", len(future_dates))

Last train date: 2025-09-05
Future start: 2025-09-06
Future end: 2025-10-31
Number of future days: 56


# Cell 12 — Tính thống kê 90 ngày gần nhất

Tính các chỉ số theo từng SKU trong 90 ngày gần nhất:
- `total_qty_90`: tổng Quantity.
- `sales_days_90`: số ngày có bán.
- `ma_90`: trung bình Quantity mỗi ngày.

Các chỉ số này dùng để nhận diện SKU không bán hoặc bán thưa.


In [12]:
window_90 = make_complete_recent_window(
    daily_data=daily,
    all_skus=all_skus,
    end_date=last_train_date,
    window_days=WINDOW_90
)

sku_stats_90 = (
    window_90
    .groupby("ItemCode")
    .agg(
        total_qty_90=("Quantity", "sum"),
        sales_days_90=("Quantity", lambda x: int((x > 0).sum())),
        ma_90=("Quantity", "mean"),
    )
    .reset_index()
)

print("sku_stats_90 shape:", sku_stats_90.shape)
display(sku_stats_90.head())

sku_stats_90 shape: (15972, 4)


,ItemCode,total_qty_90,sales_days_90,ma_90
0,SKU-00001,17.0,9,0.188889
1,SKU-00002,599.0,61,6.655556
2,SKU-00003,964.0,59,10.711111
3,SKU-00004,0.0,0,0.000000
4,SKU-00005,0.0,0,0.000000


# Cell 13 — Tính MA_28 và MA_56

Tính moving average cho từng SKU:
- `ma_28`: trung bình 28 ngày gần nhất.
- `ma_56`: trung bình 56 ngày gần nhất.

Moving average là nền tảng chính của baseline.


In [13]:
def compute_ma(daily_data, all_skus, end_date, window_days):
    """Tính moving average cho từng SKU trong window_days gần nhất."""
    window = make_complete_recent_window(
        daily_data=daily_data,
        all_skus=all_skus,
        end_date=end_date,
        window_days=window_days
    )
    
    ma = (
        window
        .groupby("ItemCode")["Quantity"]
        .mean()
        .reset_index()
        .rename(columns={"Quantity": f"ma_{window_days}"})
    )
    
    return ma

ma_28 = compute_ma(daily, all_skus, last_train_date, WINDOW_28)
ma_56 = compute_ma(daily, all_skus, last_train_date, WINDOW_56)

sku_stats = (
    sku_stats_90
    .merge(ma_28, on="ItemCode", how="left")
    .merge(ma_56, on="ItemCode", how="left")
)

print("sku_stats shape:", sku_stats.shape)
display(sku_stats.head())

sku_stats shape: (15972, 6)


,ItemCode,total_qty_90,sales_days_90,ma_90,ma_28,ma_56
0,SKU-00001,17.0,9,0.188889,0.285714,0.178571
1,SKU-00002,599.0,61,6.655556,3.500000,5.785714
2,SKU-00003,964.0,59,10.711111,12.714286,11.196429
3,SKU-00004,0.0,0,0.000000,0.000000,0.000000
4,SKU-00005,0.0,0,0.000000,0.000000,0.000000


# Cell 14 — Tính profit SKU và đánh dấu top profit

Tính tổng profit của từng SKU trên toàn bộ train.

Sau đó đánh dấu `is_top_profit` cho nhóm SKU có profit cao. Nhóm này được ưu tiên hơn vì ảnh hưởng nhiều hơn đến score leaderboard.


In [14]:
sku_profit = (
    daily
    .groupby("ItemCode", as_index=False)
    .agg(total_profit_sku=("Profit", "sum"))
)

# Chỉ dùng profit không âm để ranking
sku_profit["profit_for_rank"] = sku_profit["total_profit_sku"].clip(lower=0)

# Rank percentile: SKU càng profit cao thì rank_pct càng gần 1
sku_profit["profit_rank_pct"] = sku_profit["profit_for_rank"].rank(method="average", pct=True)

# Top profit: thuộc top 20% và profit phải dương
sku_profit["is_top_profit"] = (
    (sku_profit["profit_for_rank"] > 0) &
    (sku_profit["profit_rank_pct"] >= TOP_PROFIT_QUANTILE)
)

print("Number of top profit SKUs:", int(sku_profit["is_top_profit"].sum()))
display(sku_profit.sort_values("total_profit_sku", ascending=False).head())

Number of top profit SKUs: 3195


,ItemCode,total_profit_sku,profit_for_rank,profit_rank_pct,is_top_profit
2,SKU-00003,1.669788e+10,1.669788e+10,1.000000,True
1,SKU-00002,8.012686e+09,8.012686e+09,0.999937,True
9197,SKU-09458,2.668300e+09,2.668300e+09,0.999875,True
4,SKU-00005,2.241497e+09,2.241497e+09,0.999812,True
8354,SKU-08589,1.287112e+09,1.287112e+09,0.999750,True


# Cell 15 — Tính Weekday Average 8 tuần

Tính trung bình Quantity theo từng SKU và từng thứ trong tuần trong 8 tuần gần nhất.

Ví dụ: dự báo thứ Hai có thể dựa trên trung bình các thứ Hai gần đây của cùng SKU.


In [15]:
window_weekday = make_complete_recent_window(
    daily_data=daily,
    all_skus=all_skus,
    end_date=last_train_date,
    window_days=WEEKDAY_WINDOW
)

# pandas quy ước: 0 = Thứ Hai, 6 = Chủ Nhật
window_weekday["dayofweek"] = window_weekday["Date"].dt.dayofweek

weekday_avg = (
    window_weekday
    .groupby(["ItemCode", "dayofweek"], as_index=False)
    .agg(weekday_avg_8w=("Quantity", "mean"))
)

print("weekday_avg shape:", weekday_avg.shape)
display(weekday_avg.head())

weekday_avg shape: (111804, 3)


,ItemCode,dayofweek,weekday_avg_8w
0,SKU-00001,0,0.125
1,SKU-00001,1,0.625
2,SKU-00001,2,0.375
3,SKU-00001,3,0.125
4,SKU-00001,4,0.000


# Cell 16 — Gộp toàn bộ thống kê SKU

Gộp các bảng thống kê thành một bảng `sku_stats`.

Mỗi SKU sẽ có đủ thông tin để áp rule forecast:
- `sales_days_90`
- `total_qty_90`
- `ma_28`, `ma_56`, `ma_90`
- `total_profit_sku`
- `is_top_profit`


In [16]:
sku_stats = sku_stats.merge(
    sku_profit[["ItemCode", "total_profit_sku", "is_top_profit"]],
    on="ItemCode",
    how="left"
)

fill_values = {
    "total_qty_90": 0,
    "sales_days_90": 0,
    "ma_90": 0,
    "ma_28": 0,
    "ma_56": 0,
    "total_profit_sku": 0,
    "is_top_profit": False,
}

sku_stats = sku_stats.fillna(fill_values)
sku_stats["is_top_profit"] = sku_stats["is_top_profit"].astype(bool)

print("Final sku_stats shape:", sku_stats.shape)
display(sku_stats.head())

Final sku_stats shape: (15972, 8)


C:\Users\Louis\AppData\Local\Temp\ipykernel_29188\1696168599.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sku_stats = sku_stats.fillna(fill_values)


,ItemCode,total_qty_90,sales_days_90,ma_90,ma_28,ma_56,total_profit_sku,is_top_profit
0,SKU-00001,17.0,9,0.188889,0.285714,0.178571,3.541441e+07,True
1,SKU-00002,599.0,61,6.655556,3.500000,5.785714,8.012686e+09,True
2,SKU-00003,964.0,59,10.711111,12.714286,11.196429,1.669788e+10,True
3,SKU-00004,0.0,0,0.000000,0.000000,0.000000,8.359968e+08,True
4,SKU-00005,0.0,0,0.000000,0.000000,0.000000,2.241497e+09,True


# Cell 17 — Tạo bảng future 56 ngày

Tạo bảng các dòng cần dự báo:

```text
Date x ItemCode
```

Mỗi SKU có 56 dòng, tương ứng 56 ngày tương lai.


In [17]:
future_index = pd.MultiIndex.from_product(
    [future_dates, all_skus],
    names=["Date", "ItemCode"]
)

future = future_index.to_frame(index=False)
future["dayofweek"] = future["Date"].dt.dayofweek

print("Future shape:", future.shape)
display(future.head())

Future shape: (894432, 3)


,Date,ItemCode,dayofweek
0,2025-09-06,SKU-00001,5
1,2025-09-06,SKU-00002,5
2,2025-09-06,SKU-00003,5
3,2025-09-06,SKU-00004,5
4,2025-09-06,SKU-00005,5


# Cell 18 — Gắn thống kê quá khứ vào future

Gắn `sku_stats` và `weekday_avg_8w` vào bảng future.

Sau cell này, mỗi dòng tương lai có đủ thông tin quá khứ để tính forecast.


In [18]:
future = future.merge(sku_stats, on="ItemCode", how="left")
future = future.merge(weekday_avg, on=["ItemCode", "dayofweek"], how="left")

# Điền giá trị thiếu bằng 0
numeric_cols_to_fill = [
    "total_qty_90", "sales_days_90", "ma_90", "ma_28", "ma_56",
    "total_profit_sku", "weekday_avg_8w"
]

future[numeric_cols_to_fill] = future[numeric_cols_to_fill].fillna(0)
future["is_top_profit"] = future["is_top_profit"].fillna(False).astype(bool)

print("Future after merge shape:", future.shape)
display(future.head())

Future after merge shape: (894432, 11)


,Date,ItemCode,dayofweek,total_qty_90,sales_days_90,ma_90,ma_28,ma_56,total_profit_sku,is_top_profit,weekday_avg_8w
0,2025-09-06,SKU-00001,5,17.0,9,0.188889,0.285714,0.178571,3.541441e+07,True,0.000
1,2025-09-06,SKU-00002,5,599.0,61,6.655556,3.500000,5.785714,8.012686e+09,True,4.250
2,2025-09-06,SKU-00003,5,964.0,59,10.711111,12.714286,11.196429,1.669788e+10,True,11.125
3,2025-09-06,SKU-00004,5,0.0,0,0.000000,0.000000,0.000000,8.359968e+08,True,0.000
4,2025-09-06,SKU-00005,5,0.0,0,0.000000,0.000000,0.000000,2.241497e+09,True,0.000


# Cell 19 — Tạo forecast bằng Hybrid Rule Baseline

Áp rule forecast chính:

```text
Nếu sales_days_90 = 0 → forecast = 0
Nếu 1 <= sales_days_90 <= 3 → forecast = total_qty_90 / 90
Nếu là top profit SKU → forecast = 0.5 * ma_28 + 0.5 * weekday_avg_8w
Còn lại → forecast = 0.5 * ma_56 + 0.5 * ma_90
```

Cuối cùng clip forecast về không âm.


In [19]:
conditions = [
    # Rule 1: 90 ngày gần nhất không bán
    future["sales_days_90"] == 0,
    
    # Rule 2: SKU bán rất thưa
    (future["sales_days_90"] >= 1) & (future["sales_days_90"] <= SPARSE_SALES_DAYS_THRESHOLD),
    
    # Rule 3: SKU top profit và có bán tương đối
    (future["sales_days_90"] > SPARSE_SALES_DAYS_THRESHOLD) & (future["is_top_profit"]),
    
    # Rule 4: SKU còn lại, có bán tương đối
    (future["sales_days_90"] > SPARSE_SALES_DAYS_THRESHOLD) & (~future["is_top_profit"]),
]

choices = [
    0,
    future["total_qty_90"] / WINDOW_90,
    0.5 * future["ma_28"] + 0.5 * future["weekday_avg_8w"],
    0.5 * future["ma_56"] + 0.5 * future["ma_90"],
]

future["forecast"] = np.select(conditions, choices, default=0)

# Yêu cầu submission: forecast không âm
future["forecast"] = future["forecast"].clip(lower=0)

print("Forecast created.")
display(future[["Date", "ItemCode", "sales_days_90", "is_top_profit", "forecast"]].head())

Forecast created.


,Date,ItemCode,sales_days_90,is_top_profit,forecast
0,2025-09-06,SKU-00001,9,True,0.142857
1,2025-09-06,SKU-00002,61,True,3.875000
2,2025-09-06,SKU-00003,59,True,11.919643
3,2025-09-06,SKU-00004,0,True,0.000000
4,2025-09-06,SKU-00005,0,True,0.000000


# Cell 20 — Kiểm tra nhanh forecast

Kiểm tra forecast trước khi tạo submission:
- Không âm.
- Không NaN.
- Không vô cực.
- Không có giá trị quá bất thường.

Nếu có forecast quá lớn, xem bảng top forecast để kiểm tra SKU đó.


In [20]:
print("Forecast summary:")
display(future["forecast"].describe())

print("Number of negative forecasts:", int((future["forecast"] < 0).sum()))
print("Number of NaN forecasts:", int(future["forecast"].isna().sum()))
print("Number of infinite forecasts:", int(np.isinf(future["forecast"].values).sum()))

print("\nTop 20 highest forecasts:")
display(
    future[["Date", "ItemCode", "sales_days_90", "is_top_profit", "ma_28", "ma_56", "ma_90", "weekday_avg_8w", "forecast"]]
    .sort_values("forecast", ascending=False)
    .head(20)
)

Forecast summary:


count    894432.000000
mean          0.082523
std           1.049562
min           0.000000
25%           0.000000
50%           0.000000
75%           0.011111
max         108.125000
Name: forecast, dtype: float64

Number of negative forecasts: 0
Number of NaN forecasts: 0
Number of infinite forecasts: 0

Top 20 highest forecasts:


,Date,ItemCode,sales_days_90,is_top_profit,ma_28,ma_56,ma_90,weekday_avg_8w,forecast
840036,2025-10-28,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
281016,2025-09-23,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
392820,2025-09-30,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
504624,2025-10-07,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
57408,2025-09-09,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
169212,2025-09-16,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
616428,2025-10-14,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
728232,2025-10-21,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
536568,2025-10-09,SKU-09760,69,True,83.5,80.5,98.188889,130.00,106.750
871980,2025-10-30,SKU-09760,69,True,83.5,80.5,98.188889,130.00,106.750


# Cell 21 — Gắn số thứ tự ngày dự báo

Tạo cột `horizon` cho từng SKU:

```text
1 → ngày dự báo đầu tiên
...
56 → ngày dự báo cuối cùng
```

Cột này dùng để chuyển forecast sang `F1` đến `F28`.


In [21]:
future = future.sort_values(["ItemCode", "Date"]).copy()
future["horizon"] = future.groupby("ItemCode").cumcount() + 1

print("Horizon range:", future["horizon"].min(), "→", future["horizon"].max())
display(future[["Date", "ItemCode", "horizon", "forecast"]].head())

Horizon range: 1 → 56


,Date,ItemCode,horizon,forecast
0,2025-09-06,SKU-00001,1,0.142857
15972,2025-09-07,SKU-00001,2,0.142857
31944,2025-09-08,SKU-00001,3,0.205357
47916,2025-09-09,SKU-00001,4,0.455357
63888,2025-09-10,SKU-00001,5,0.330357


# Cell 22 — Tách validation và evaluation

Chia 56 ngày forecast thành 2 phần:
- `future_validation`: 28 ngày đầu, dùng cho Public leaderboard.
- `future_evaluation`: 28 ngày sau, dùng cho Private leaderboard.

Evaluation được đổi lại horizon về 1–28 để khớp cột `F1`–`F28`.


In [22]:
future_validation = future[future["horizon"] <= PUBLIC_HORIZON].copy()
future_evaluation = future[future["horizon"] > PUBLIC_HORIZON].copy()

# Evaluation là ngày 29-56, nhưng trong file submission vẫn phải ghi vào F1-F28
future_evaluation["horizon"] = future_evaluation["horizon"] - PUBLIC_HORIZON

print("future_validation shape:", future_validation.shape)
print("future_evaluation shape:", future_evaluation.shape)
print("Validation horizon:", future_validation["horizon"].min(), "→", future_validation["horizon"].max())
print("Evaluation horizon:", future_evaluation["horizon"].min(), "→", future_evaluation["horizon"].max())

future_validation shape: (447216, 13)
future_evaluation shape: (447216, 13)
Validation horizon: 1 → 28
Evaluation horizon: 1 → 28


# Cell 23 — Chuyển forecast sang format F1–F28

Chuyển từ long format:

```text
Date | ItemCode | horizon | forecast
```

sang wide format Kaggle cần:

```text
id | F1 | F2 | ... | F28
```


In [23]:
forecast_cols = [f"F{i}" for i in range(1, PUBLIC_HORIZON + 1)]

validation_wide = (
    future_validation
    .pivot(index="ItemCode", columns="horizon", values="forecast")
    .reset_index()
)

validation_wide.columns = ["ItemCode"] + forecast_cols
validation_wide["id"] = validation_wide["ItemCode"] + "_validation"
validation_wide = validation_wide[["id"] + forecast_cols]


evaluation_wide = (
    future_evaluation
    .pivot(index="ItemCode", columns="horizon", values="forecast")
    .reset_index()
)

evaluation_wide.columns = ["ItemCode"] + forecast_cols
evaluation_wide["id"] = evaluation_wide["ItemCode"] + "_evaluation"
evaluation_wide = evaluation_wide[["id"] + forecast_cols]

print("validation_wide shape:", validation_wide.shape)
print("evaluation_wide shape:", evaluation_wide.shape)

display(validation_wide.head())
display(evaluation_wide.head())

validation_wide shape: (15972, 29)
evaluation_wide shape: (15972, 29)


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857,0.142857,0.142857,...,0.330357,0.205357,0.142857,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857
1,SKU-00002_validation,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000,3.875000,1.750000,...,6.437500,5.750000,5.125000,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000
2,SKU-00003_validation,11.919643,6.357143,12.544643,11.919643,14.982143,14.107143,11.857143,11.919643,6.357143,...,14.982143,14.107143,11.857143,11.919643,6.357143,12.544643,11.919643,14.982143,14.107143,11.857143
3,SKU-00004_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,SKU-00005_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_evaluation,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857,0.142857,0.142857,...,0.330357,0.205357,0.142857,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857
1,SKU-00002_evaluation,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000,3.875000,1.750000,...,6.437500,5.750000,5.125000,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000
2,SKU-00003_evaluation,11.919643,6.357143,12.544643,11.919643,14.982143,14.107143,11.857143,11.919643,6.357143,...,14.982143,14.107143,11.857143,11.919643,6.357143,12.544643,11.919643,14.982143,14.107143,11.857143
3,SKU-00004_evaluation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,SKU-00005_evaluation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


# Cell 24 — Gộp thành submission theo sample_submission

Gộp `validation_wide` và `evaluation_wide`, sau đó merge với `sample_sub[["id"]]`.

Mục tiêu: giữ đúng danh sách id, số dòng và thứ tự như sample submission.


In [24]:
submission_pred = pd.concat(
    [validation_wide, evaluation_wide],
    axis=0,
    ignore_index=True
)

submission = sample_sub[["id"]].merge(submission_pred, on="id", how="left")

print("submission_pred shape:", submission_pred.shape)
print("submission shape:", submission.shape)

display(submission.head())
display(submission.tail())

submission_pred shape: (31944, 29)
submission shape: (31944, 29)


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857,0.142857,0.142857,...,0.330357,0.205357,0.142857,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857
1,SKU-00002_validation,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000,3.875000,1.750000,...,6.437500,5.750000,5.125000,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000
2,SKU-00003_validation,11.919643,6.357143,12.544643,11.919643,14.982143,14.107143,11.857143,11.919643,6.357143,...,14.982143,14.107143,11.857143,11.919643,6.357143,12.544643,11.919643,14.982143,14.107143,11.857143
3,SKU-00004_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,SKU-00005_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
31939,SKU-16329_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31940,SKU-16330_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31941,SKU-16331_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31942,SKU-16332_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31943,SKU-16333_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Cell 25 — Kiểm tra submission trước khi lưu

Các điều kiện cần đạt:

```text
Correct columns: True
Correct number of rows: True
Duplicate ids: 0
NaN values: 0
Negative values: 0
Infinite values: 0
```

Nếu một điều kiện sai, chưa nên nộp Kaggle.


In [25]:
expected_cols = ["id"] + forecast_cols

# Lớp bảo vệ cuối cùng trước khi kiểm tra
submission[forecast_cols] = submission[forecast_cols].fillna(0)
submission[forecast_cols] = submission[forecast_cols].clip(lower=0)

print("Correct columns:", submission.columns.tolist() == expected_cols)
print("Correct number of rows:", submission.shape[0] == sample_sub.shape[0])
print("Duplicate ids:", int(submission["id"].duplicated().sum()))
print("NaN values:", int(submission.isna().sum().sum()))
print("Negative values:", int((submission[forecast_cols] < 0).sum().sum()))
print("Infinite values:", int(np.isinf(submission[forecast_cols].values).sum()))

print("\nSubmission preview:")
display(submission.head())

Correct columns: True
Correct number of rows: True
Duplicate ids: 0
NaN values: 0
Negative values: 0
Infinite values: 0

Submission preview:


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857,0.142857,0.142857,...,0.330357,0.205357,0.142857,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857
1,SKU-00002_validation,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000,3.875000,1.750000,...,6.437500,5.750000,5.125000,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000
2,SKU-00003_validation,11.919643,6.357143,12.544643,11.919643,14.982143,14.107143,11.857143,11.919643,6.357143,...,14.982143,14.107143,11.857143,11.919643,6.357143,12.544643,11.919643,14.982143,14.107143,11.857143
3,SKU-00004_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,SKU-00005_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


# Cell 26 — Lưu file submission

Lưu file CSV cuối cùng vào folder `outputs/`.

File output là:

```text
submission_baseline_hybrid_v01.csv
```

Chỉ nộp file này nếu cell kiểm tra trước đó đạt yêu cầu.


In [26]:
submission.to_csv(SUBMISSION_PATH, index=False)

print("Saved submission to:", SUBMISSION_PATH)
print("File exists:", SUBMISSION_PATH.exists())

Saved submission to: outputs\submission_baseline_hybrid_v01.csv
File exists: True


# Cell 27 — Ghi log kết quả

**Version:** `baseline_hybrid_v01`

**Data handling:**
- Convert `Date` và các cột số cần dùng.
- Tạo `Profit`.
- Gom transaction về daily sales.
- Clip daily `Quantity` âm về 0.
- Chỉ tạo full table cho các window gần nhất để nhẹ máy.

**Rule forecast:**
- Không bán 90 ngày → 0.
- Bán 1–3 ngày trong 90 ngày → `total_qty_90 / 90`.
- Top profit SKU → `0.5 * ma_28 + 0.5 * weekday_avg_8w`.
- SKU còn lại → `0.5 * ma_56 + 0.5 * ma_90`.

**Public LB score:** điền sau khi submit.
